## OpenAPI Tools with Foundry Agent Service  

### Installing Required Libraries

In [1]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity jsonref

Note: you may need to restart the kernel to use updated packages.


### Setting Up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
)
import jsonref

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

### Setting Up the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Initializing the OpenAPI Tool

In [ ]:
with open("./weather_openapi.json", "r") as f:
        openapi_weather = jsonref.loads(f.read())

# Initialize agent OpenApi tool using the read in OpenAPI spec
weather_tool = {
        "type": "openapi",
        "openapi":{
            "name": "weather",
            "spec": openapi_weather,
            "auth": {
                "type": "anonymous"
            },
        }
}

### Setting Up Our Agent with OpenAPI Tool Spec

In [5]:
agent_name = "weather-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful Weather Agent that provides weather information using the provided OpenAPI Tool",
        tools=[weather_tool]
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: weather-agent:1, name: weather-agent, version: 1)


### Creating a Conversation Object for the Agent Chat System

In [6]:
# creating an openai client first
openai_client = project_client.get_openai_client()

# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_21e8f0932426b88200mF6VOzya6Y23FhMUBYHdV8F4x37PdX9r


In [9]:
user_input = "What's the weather like in Bangalore today?"

### Chat with The Agent

In [10]:
response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_input
        )
print(f"Weather Agent: {response.output_text}")

Weather Agent: Today's weather in Bangalore is sunny with a current temperature of 26°C (79°F). The wind is coming from the south-southeast at 9 km/h (6 mph), and the humidity is at 58%. No precipitation is expected today. 

The maximum temperature will reach 33°C (91°F), while the minimum will be 20°C (68°F). Enjoy your day!
